# Eksperimen 0: Baseline Leksikal (TF-IDF) & Semantik (Frozen SBERT)

Baseline leksikal berbasis TF-IDF dan semantik berbasis Sentence-BERT (SBERT). Model regresi menggunakan Support Vector Regression (SVR) untuk memprediksi skor jawaban siswa. Eksperimen ini menetapkan batas performa bawah sebelum pengujian model fine-tuning IndoBERT dan late fusion.

## 0. Persiapan Lingkungan & Konfigurasi

In [ ]:
import sys
import os

try:
    import google.colab
    IN_COLAB = True
    print("[Info] Menjalankan notebook di lingkungan Google Colab.")
    if not os.path.exists("/content/IndoASAG"):
        try:
            from google.colab import userdata
            GH_TOKEN = userdata.get('GITHUB_TOKEN')
        except Exception:
            GH_TOKEN = None
            
        if GH_TOKEN:
            repo_url = f"https://{GH_TOKEN}@github.com/Rnov24/IndoASAG.git"
        else:
            repo_url = "https://github.com/Rnov24/IndoASAG.git"
            
        os.system(f"git clone {repo_url} /content/IndoASAG")
    os.chdir("/content/IndoASAG")
    os.system("pip install -q -r requirements.txt")
    REPO_ROOT = "/content/IndoASAG"
except ImportError:
    IN_COLAB = False
    try:
        REPO_ROOT = os.path.abspath(os.path.join(os.path.dirname(__file__), ".."))
    except NameError:
        REPO_ROOT = os.path.abspath(os.getcwd())
    print(f"[Info] Menjalankan notebook di lingkungan lokal: {REPO_ROOT}")

sys.path.insert(0, os.path.join(REPO_ROOT, "src"))

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity, euclidean_distances, pairwise_distances
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sentence_transformers import SentenceTransformer

from evaluation import calculate_metrics

DATASET_DIR = os.path.join(REPO_ROOT, "dataset")
RESULTS_DIR = os.path.join(REPO_ROOT, "results")
PREDS_DIR = os.path.join(RESULTS_DIR, "predictions")

for d in [PREDS_DIR]:
    os.makedirs(d, exist_ok=True)

## 1. Pemuatan Dataset

In [ ]:
train_df = pd.read_parquet(os.path.join(DATASET_DIR, "train.parquet"))
test_df = pd.read_parquet(os.path.join(DATASET_DIR, "test.parquet"))

print(f"Data Train : {len(train_df)} baris")
print(f"Data Test  : {len(test_df)} baris")

SEEDS = [42, 100, 2026, 777, 999]
FOLDS = 5

y_true_train = train_df["score_norm"].values
y_true_test = test_df["score_norm"].values

## 2. Eksperimen Baseline Leksikal (TF-IDF + Jarak Klasik + SVR)

Ekstraksi kemiripan leksikal berbasis TF-IDF (Cosine, Euclidean) dan Jaccard Overlap dari representasi kata. Fitur jarak dikonkatenasi untuk pemodelan SVR.

In [ ]:
print("="*60)
print("MEMULAI EKSPERIMEN BASELINE LEKSIKAL (TF-IDF)")
print("="*60)

# Menggabungkan teks untuk pelatihan TF-IDF
all_texts = pd.concat([train_df["student_answer"], train_df["reference_answer"]])
tfidf = TfidfVectorizer(max_features=5000, sublinear_tf=True)
tfidf.fit(all_texts)

cv = CountVectorizer(max_features=5000, binary=True)
cv.fit(all_texts)

def extract_lexical_features(df_subset):
    va_t = tfidf.transform(df_subset["student_answer"])
    vr_t = tfidf.transform(df_subset["reference_answer"])
    va_c = cv.transform(df_subset["student_answer"])
    vr_c = cv.transform(df_subset["reference_answer"])
    
    cos = np.array([cosine_similarity(va_t[i], vr_t[i])[0, 0] for i in range(va_t.shape[0])])
    euc = np.array([euclidean_distances(va_t[i], vr_t[i])[0, 0] for i in range(va_t.shape[0])])
    
    jac = []
    for i in range(va_c.shape[0]):
        if va_c[i].nnz > 0 or vr_c[i].nnz > 0:
            jac.append(1.0 - pairwise_distances(va_c[i].toarray(), vr_c[i].toarray(), metric="jaccard")[0, 0])
        else:
            jac.append(1.0)
    return np.column_stack([cos, euc, np.array(jac)])

print("Mengekstrak fitur leksikal untuk Train dan Test set...")
X_train_lex = extract_lexical_features(train_df)
X_test_lex = extract_lexical_features(test_df)

# Eksekusi Cross-Validation (OOF) & Held-Out Test untuk 5 Seeds
for seed in SEEDS:
    kf = KFold(n_splits=FOLDS, shuffle=True, random_state=seed)
    oof_predictions = np.zeros(len(train_df))
    
    for fold, (train_idx, val_idx) in enumerate(kf.split(train_df)):
        X_tr, y_tr = X_train_lex[train_idx], y_true_train[train_idx]
        X_va = X_train_lex[val_idx]
        
        sc = StandardScaler()
        X_tr_s = sc.fit_transform(X_tr)
        X_va_s = sc.transform(X_va)
        
        svr = SVR(kernel="rbf", C=10.0, epsilon=0.05)
        svr.fit(X_tr_s, y_tr)
        oof_predictions[val_idx] = np.clip(svr.predict(X_va_s), 0.0, 1.0)
        
    # Latih pada seluruh Train Set untuk held-out test predictions
    sc_full = StandardScaler()
    X_train_s = sc_full.fit_transform(X_train_lex)
    X_test_s = sc_full.transform(X_test_lex)
    
    svr_full = SVR(kernel="rbf", C=10.0, epsilon=0.05)
    svr_full.fit(X_train_s, y_true_train)
    test_predictions = np.clip(svr_full.predict(X_test_s), 0.0, 1.0)
    
    # Simpan hasil prediksi
    np.save(os.path.join(PREDS_DIR, f"exp1_lexical_oof_seed{seed}.npy"), oof_predictions)
    np.save(os.path.join(PREDS_DIR, f"exp1_lexical_test_seed{seed}.npy"), test_predictions)
    
    metrics = calculate_metrics(y_true_test, test_predictions)
    print(f"[Seed {seed}] Leksikal QWK: {metrics['QWK']:.4f} | Pearson: {metrics['Pearson']:.4f}")

## 3. Eksperimen Baseline Semantik (Frozen SBERT + Metrik Jarak + SVR)

Ekstraksi representasi semantik menggunakan model Sentence-BERT multibahasa tanpa fine-tuning (frozen). Metrik jarak yang dihitung meliputi kemiripan Cosine, jarak Euclidean, dan Jaccard kontinu (Tanimoto).

In [ ]:
print("\n" + "="*60)
print("MEMULAI EKSPERIMEN BASELINE SEMANTIK (FROZEN SBERT)")
print("="*60)

sbert_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

print("Mengubah jawaban menjadi embedding (encoding SBERT)...")
train_ans_emb = sbert_model.encode(train_df["student_answer"].tolist(), batch_size=64, normalize_embeddings=True, show_progress_bar=False)
train_ref_emb = sbert_model.encode(train_df["reference_answer"].tolist(), batch_size=64, normalize_embeddings=True, show_progress_bar=False)

test_ans_emb = sbert_model.encode(test_df["student_answer"].tolist(), batch_size=64, normalize_embeddings=True, show_progress_bar=False)
test_ref_emb = sbert_model.encode(test_df["reference_answer"].tolist(), batch_size=64, normalize_embeddings=True, show_progress_bar=False)

def extract_sbert_features(ans_emb, ref_emb):
    cos_sim = np.sum(ans_emb * ref_emb, axis=1)
    euc_dist = np.linalg.norm(ans_emb - ref_emb, axis=1)
    tanimoto_jac = cos_sim / (2.0 - cos_sim)
    return np.column_stack([cos_sim, euc_dist, tanimoto_jac])

X_train_sbert = extract_sbert_features(train_ans_emb, train_ref_emb)
X_test_sbert = extract_sbert_features(test_ans_emb, test_ref_emb)

# Eksekusi Cross-Validation (OOF) & Held-Out Test untuk 5 Seeds
for seed in SEEDS:
    kf = KFold(n_splits=FOLDS, shuffle=True, random_state=seed)
    oof_predictions = np.zeros(len(train_df))
    
    for fold, (train_idx, val_idx) in enumerate(kf.split(train_df)):
        X_tr, y_tr = X_train_sbert[train_idx], y_true_train[train_idx]
        X_va = X_train_sbert[val_idx]
        
        sc = StandardScaler()
        X_tr_s = sc.fit_transform(X_tr)
        X_va_s = sc.transform(X_va)
        
        svr = SVR(kernel="rbf", C=10.0, epsilon=0.05)
        svr.fit(X_tr_s, y_tr)
        oof_predictions[val_idx] = np.clip(svr.predict(X_va_s), 0.0, 1.0)
        
    # Latih pada seluruh Train Set untuk held-out test predictions
    sc_full = StandardScaler()
    X_train_s = sc_full.fit_transform(X_train_sbert)
    X_test_s = sc_full.transform(X_test_sbert)
    
    svr_full = SVR(kernel="rbf", C=10.0, epsilon=0.05)
    svr_full.fit(X_train_s, y_true_train)
    test_predictions = np.clip(svr_full.predict(X_test_s), 0.0, 1.0)
    
    # Simpan hasil prediksi
    np.save(os.path.join(PREDS_DIR, f"exp2_sbert_oof_seed{seed}.npy"), oof_predictions)
    np.save(os.path.join(PREDS_DIR, f"exp2_sbert_test_seed{seed}.npy"), test_predictions)
    
    metrics = calculate_metrics(y_true_test, test_predictions)
    print(f"[Seed {seed}] SBERT QWK: {metrics['QWK']:.4f} | Pearson: {metrics['Pearson']:.4f}")

print("\n[OK] Seluruh eksperimen baseline selesai dan berhasil disimpan.")